# Hallmark: data servers, the catalogue and the object store

This notebook runs every case of Hallmark's data backend against servers it
starts on this computer, so it needs no account and no network:

- an HTTP archive that publishes a `sha256sums` file,
- a mirror of that archive,
- a WebDAV store that reports sizes and its own checksums,
- an HTTP server that requires a token,
- an S3 bucket, kept in memory by `moto` (skipped when `boto3` and `moto` are
  not installed).

After each step, `show()` prints what changed: what is staged in `.hm`, how
many files `.hm/objects` holds, and each data file with its
permissions and hard-link count. [doc/data_backend.md](../doc/data_backend.md)
explains the design behind every step.

## Setup

Everything happens in a temporary folder. Environment variables point Hallmark
and boto3 at files in that folder, so your own `~/.config/hallmark/auth.yml`
and AWS settings are never read.

In [1]:
import hashlib
import os
import shutil
import stat
import tempfile
from email.utils import formatdate
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from threading import Thread
from urllib.parse import quote

import yaml

from hallmark import Repo

workspace = Path(tempfile.mkdtemp(prefix="hallmark-demo-"))
os.chdir(workspace)
os.environ.update({
    "HALLMARK_AUTH_FILE": str(workspace / "auth.yml"),
    "XDG_CONFIG_HOME": str(workspace / "config"),
    "NETRC": str(workspace / "no-netrc"),
    "AWS_CONFIG_FILE": str(workspace / "aws-config"),
    "AWS_SHARED_CREDENTIALS_FILE": str(workspace / "aws-credentials"),
    "AWS_ACCESS_KEY_ID": "demo",
    "AWS_SECRET_ACCESS_KEY": "demo",
    "AWS_DEFAULT_REGION": "us-east-1",
})
os.environ.pop("AWS_PROFILE", None)

The servers' files: an archive of M87 and Sgr A\* data with a checksum
manifest, a mirror holding a copy of it, calibration tables on WebDAV, and
private observing notes.

In [2]:
def publish_sums(folder):
    """Write the sha256sums manifest the archive publishes beside its files."""
    lines = [f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.name}"
             for path in sorted(folder.glob("*.txt"))]
    (folder / "sha256sums").write_text("\n".join(lines) + "\n")


archive = workspace / "servers/archive"
archive.mkdir(parents=True)
for name in ["M87_001.txt", "M87_002.txt", "SgrA_001.txt", "SgrA_002.txt"]:
    (archive / name).write_text(f"EHT data {name}\n")
publish_sums(archive)
mirror = workspace / "servers/mirror"
shutil.copytree(archive, mirror)

calib = workspace / "servers/dav/calib"
calib.mkdir(parents=True)
for day in (1, 2):
    (calib / f"gains_{day:03d}.txt").write_text(f"gain table {day}\n")

private = workspace / "servers/private"
private.mkdir()
(private / "notes_001.txt").write_text("observing notes\n")

16

One small server class serves a folder. It can require a bearer token, and it
answers WebDAV `PROPFIND` requests with each file's size, date and SHA-1, as
ownCloud and CyVerse do. It also records the files each server sent, so the
demo can show where a download came from.

In [3]:
class DemoServer(SimpleHTTPRequestHandler):
    token = None
    sent = None

    def log_message(self, *args):
        pass

    def allowed(self):
        if self.token and self.headers.get("Authorization") != f"Bearer {self.token}":
            self.send_response(401)
            self.send_header("Content-Length", "0")
            self.end_headers()
            return False
        return True

    def do_GET(self):
        if self.allowed():
            if not self.path.endswith("/"):
                self.sent.append(self.path)
            super().do_GET()

    def do_HEAD(self):
        if self.allowed():
            super().do_HEAD()

    def do_PROPFIND(self):
        folder = Path(self.translate_path(self.path))
        if not folder.is_dir():
            self.send_error(404)
            return
        base = self.path if self.path.endswith("/") else self.path + "/"
        responses = []
        for entry in [folder, *sorted(folder.iterdir())]:
            if entry.is_dir():
                href = base if entry == folder else base + quote(entry.name) + "/"
                props = "<d:resourcetype><d:collection/></d:resourcetype>"
            else:
                data = entry.read_bytes()
                href = base + quote(entry.name)
                props = (f"<d:resourcetype/><d:getcontentlength>{len(data)}"
                         "</d:getcontentlength><oc:checksums><oc:checksum>"
                         f"SHA1:{hashlib.sha1(data).hexdigest()}"
                         "</oc:checksum></oc:checksums>")
            props += ("<d:getlastmodified>"
                      f"{formatdate(entry.stat().st_mtime, usegmt=True)}"
                      "</d:getlastmodified>")
            responses.append(
                f"<d:response><d:href>{href}</d:href><d:propstat><d:prop>{props}"
                "</d:prop><d:status>HTTP/1.1 200 OK</d:status></d:propstat>"
                "</d:response>")
        body = ('<?xml version="1.0"?><d:multistatus xmlns:d="DAV:" '
                'xmlns:oc="http://owncloud.org/ns">' + "".join(responses)
                + "</d:multistatus>").encode()
        self.send_response(207)
        self.send_header("Content-Type", "application/xml")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


servers, sent = {}, {}


def start(name, folder, *, token=None, port=0):
    """Serve a folder on this computer and return its URL."""
    sent[name] = []
    handler = type(name, (DemoServer,), {"token": token, "sent": sent[name]})
    server = ThreadingHTTPServer(("127.0.0.1", port),
                                 partial(handler, directory=str(folder)))
    Thread(target=server.serve_forever, daemon=True).start()
    servers[name] = server
    return f"http://127.0.0.1:{server.server_port}"


def stop(name):
    servers[name].shutdown()
    servers[name].server_close()


TOKEN = "demo-token-123"
ARCHIVE = start("archive", archive)
MIRROR = start("mirror", mirror)
DAV = start("dav", workspace / "servers/dav").replace("http://", "webdav://")
PRIVATE = start("private", private, token=TOKEN)
print(ARCHIVE, MIRROR, DAV, PRIVATE, sep="\n")

http://127.0.0.1:38107
http://127.0.0.1:45507
webdav://127.0.0.1:37061
http://127.0.0.1:34295


The S3 bucket lives in memory. Its objects are uploaded with SHA-256
checksums, which S3 reports back to Hallmark.

In [4]:
try:
    import boto3
    from moto import mock_aws
except ImportError:
    s3_mock = None
    print("boto3 and moto are not installed, so the S3 cells are skipped.")
else:
    s3_mock = mock_aws()
    s3_mock.start()
    bucket = boto3.client("s3", region_name="us-east-1")
    bucket.create_bucket(Bucket="ehtc-sims")
    for day in (1, 2):
        bucket.put_object(Bucket="ehtc-sims", Key=f"m87/sim_{day:03d}.h5",
                          Body=f"simulated image {day}\n".encode(),
                          ChecksumAlgorithm="SHA256")
    print("S3 bucket ehtc-sims is ready")

S3 bucket ehtc-sims is ready


Two helpers: `show()` prints a repository's state, and `expect_error()` runs
something Hallmark should refuse and prints the reason.

In [5]:
def count_files(folder):
    """Count stored objects, leaving out stat records and unfinished downloads."""
    if not folder.exists():
        return 0
    return sum(1 for path in folder.rglob("*") if path.is_file()
               and path.relative_to(folder).parts[0] not in {"stat", "tmp"})


def show(repo):
    """Print what is staged in .hm, the two stores, and the data files."""
    staged = repo.dothm.git.diff("--cached", "--name-status").splitlines()
    records = sum(line.split("\t")[-1].startswith("versions/") for line in staged)
    files = [line.replace("\t", " ") for line in staged
             if not line.split("\t")[-1].startswith("versions")]
    print("staged in .hm:", ", ".join(files) or "nothing",
          f"(and {records} row record(s) in versions/)" if records else "")
    hm = repo.dothm.path
    print(f"objects: {count_files(hm / 'objects')} file(s)")
    for path in sorted(repo.worktree.rglob("*")):
        relative = path.relative_to(repo.worktree)
        if path.is_file() and relative.parts[0] != ".hm":
            info = path.stat()
            print(f"  {relative.as_posix():20} {stat.filemode(info.st_mode)}"
                  f"  links={info.st_nlink}")


def expect_error(action):
    """Run something Hallmark should refuse, and print the reason."""
    try:
        action()
    except Exception as exc:
        print(f"{type(exc).__name__}: {exc}")
    else:
        raise AssertionError("Hallmark accepted this; the demo expected a refusal")

## 1. A new repository

`Repo.init` creates the data folder with a `.hm` git repository inside. Git
tracks the catalogue's text files. `objects/` appears later and is
never added to git.

In [6]:
lab = Repo.init("lab")
lab.set_identity("Hallmark Demo", "demo@example.invalid")
print(sorted(path.name for path in lab.dothm.path.iterdir()))
print(lab.dothm.git.log("--oneline"))
show(lab)

['.git', 'README.md', 'config.yml', 'data.tsv', 'meta.yml']
00c3537 Initial commit: local `.hm` repository
staged in .hm: A config.yml, A data.tsv, A meta.yml 
objects: 0 file(s)


`init` commits only a README; the three state files are staged, not committed.
The commit author set with `set_identity` (`hm config user.name` on the command
line) is stored in `.hm/.git/config` and is never shared.

## 2. Local files and `.hm/objects`

This repository catalogues files you already have. `add` hashes each file and
stages one row for it. Nothing is copied yet.

In [7]:
for n in (1, 2):
    (lab.worktree / f"run_{n}.txt").write_text(f"result {n}\n")
lab.add("run_{n}.txt")
print(lab.state.data)
show(lab)

                                       sha1       path  n
0  fb042ff701f43a02456fa71ce4243fafad3e6e32  run_1.txt  1
1  1b5349c3c3f0a66ceb46a35e93a498d53fe19daf  run_2.txt  2
staged in .hm: A config.yml, A data.tsv, A meta.yml 
objects: 0 file(s)
  run_1.txt            -rw-r--r--  links=1
  run_2.txt            -rw-r--r--  links=1


`commit` copies each new file into `.hm/objects`, named by its SHA-1. These
copies are the only record of old versions, so never delete them.

In [8]:
lab.commit("Add two runs")
show(lab)

staged in .hm: nothing 
objects: 2 file(s)
  run_1.txt            -rw-r--r--  links=1
  run_2.txt            -rw-r--r--  links=1


An edited file shows up in `status`. Adding and committing it stores one more
object, and the old one stays.

In [9]:
(lab.worktree / "run_1.txt").write_text("result 1, recalibrated\n")
print(lab.status()["worktree"])
lab.add("run_1.txt")
lab.commit("Recalibrate run 1")
show(lab)

{'modified': ['run_1.txt'], 'deleted': []}


staged in .hm: nothing 
objects: 3 file(s)
  run_1.txt            -rw-r--r--  links=1
  run_2.txt            -rw-r--r--  links=1


`checkout` rewrites tracked files by copying them out of `objects`. A file
added on the `exp` branch disappears on `main` and comes back on `exp`.

In [10]:
lab.create_branch("exp")
lab.checkout("exp")
(lab.worktree / "run_3.txt").write_text("result 3\n")
lab.add("run_3.txt")
lab.commit("Add run 3 on exp")
for branch in ("main", "exp", "main"):
    lab.checkout(branch)
    print(f"on {branch}:", sorted(path.name for path in lab.worktree.glob("*.txt")))

on main: ['run_1.txt', 'run_2.txt']


on exp: ['run_1.txt', 'run_2.txt', 'run_3.txt']


on main: ['run_1.txt', 'run_2.txt']


Objects never leave this computer, so a repository of local files cannot be
cloned.

In [11]:
expect_error(lambda: Repo.clone("lab", "lab-copy", download=False))

CloneError: This repository tracks local files. Their contents aren't included in a clone.
To share data, put it on a server and catalog it with hm add URL, or copy the data folder yourself.


## 3. Catalogue a data server

`add` with a URL lists the server and stages one row per matching file;
nothing is downloaded. The archive publishes `sha256sums`, so each row gets a
checksum whose `checksum_source` is `published`. Python's small web server
lists no sizes or dates, so those columns stay empty.

In [12]:
eht = Repo.init("eht")
eht.set_identity("Hallmark Demo", "demo@example.invalid")
eht.add(ARCHIVE + "/M87_{day:03d}.txt", source="m87")
eht.state.data

,path,checksum_algorithm,checksum,size_bytes,mtime,checksum_source,source,day
0,M87_001.txt,sha256,4c16fd9e2cdebfdb8a5ecf18ca0f34958be7a22600fa11...,,,published,m87,001
1,M87_002.txt,sha256,a4dec2ee71577559ea5f2ab21d0f3edb710d9113089616...,,,published,m87,002


The configuration records the *data remote* (the server root, here named
`origin`) and the *source* (what was selected under it). Both are committed
with the catalogue.

In [13]:
print(yaml.safe_dump(eht.state.config, sort_keys=False))

data:
- db: data.tsv
remote:
- name: origin
  url: http://127.0.0.1:38107/
  backend: http
sources:
- name: m87
  remote: origin
  fmt: M87_{day:03d}.txt



The catalogue itself is `.hm/data.tsv`: one line per file, sorted by path, so
git shows each row's changes as a line of its own.

In [14]:
print((eht.dothm.path / "data.tsv").read_text())
show(eht)
eht.commit("Catalogue M87 from the archive")

path	checksum_algorithm	checksum	size_bytes	mtime	checksum_source	source	day
M87_001.txt	sha256	4c16fd9e2cdebfdb8a5ecf18ca0f34958be7a22600fa11e5e7233ed98290fa5f			published	m87	001
M87_002.txt	sha256	a4dec2ee71577559ea5f2ab21d0f3edb710d9113089616edeea3dddf57b58fa2			published	m87	002

staged in .hm: A config.yml, A data.tsv, A meta.yml 
objects: 0 file(s)


True

## 4. Several sources on one server

Another `add` creates another source. Paths must stay unique across sources,
so cataloguing the same files a second time is refused...

In [15]:
eht.add(ARCHIVE + "/SgrA_{day:03d}.txt", source="sgra")
expect_error(lambda: eht.add(ARCHIVE, filter="M87_*", source="copy"))

ValueError: 2 path(s) already belong to another source:
  M87_001.txt
  M87_002.txt
To change the pattern of source 'm87', add again with --source m87.


...unless the new source gets its own local folder with `prefix`. This is a
second, separately catalogued copy of the same files. It is not a mirror
(section 11).

In [16]:
eht.add(ARCHIVE, filter="M87_*", source="copy", prefix="copy")
eht.state.data[["path", "source", "checksum_source"]]

,path,source,checksum_source
0,M87_001.txt,m87,published
1,M87_002.txt,m87,published
2,SgrA_001.txt,sgra,published
3,SgrA_002.txt,sgra,published
4,copy/M87_001.txt,copy,published
5,copy/M87_002.txt,copy,published


## 5. Several servers

### WebDAV

`webdav://` selects the WebDAV backend. Its listing reports sizes, dates and
the server's own checksums, so `checksum_source` is `server`. A new server
becomes a new data remote, named here with `remote`.

In [17]:
eht.add(DAV + "/calib/", source="calib", remote="dav", prefix="calib")
columns = ["path", "size_bytes", "mtime", "checksum_algorithm", "checksum_source"]
eht.state.data.loc[eht.state.data.source == "calib", columns]

,path,size_bytes,mtime,checksum_algorithm,checksum_source
4,calib/gains_001.txt,13,1791877490,sha1,server
5,calib/gains_002.txt,13,1791877490,sha1,server


### S3

An S3 prefix is catalogued the same way. Objects uploaded with SHA-256
checksums report them, again with `checksum_source` `server`.

In [18]:
if s3_mock:
    eht.add("s3://ehtc-sims/m87/", source="sims", remote="s3", prefix="sims")
    print(eht.state.data.loc[eht.state.data.source == "sims", columns])

              path size_bytes       mtime checksum_algorithm checksum_source
8  sims/sim_001.h5         18  1791877490             sha256          server
9  sims/sim_002.h5         18  1791877490             sha256          server


### A server that needs a login

Without credentials the server answers 401, and nothing is catalogued.

In [19]:
expect_error(lambda: eht.add(PRIVATE, source="notes", remote="private"))

DownloadError: Remote scan failed (previous catalogue kept): Failed to download from http://127.0.0.1: HTTP 401; name an auth profile with --auth, or use ~/.netrc


Credentials live in a per-user `auth.yml`, never in the repository. A profile
lists the hosts it may be sent to and says where its secret comes from, here
an environment variable.

In [20]:
os.environ["DEMO_TOKEN"] = TOKEN
auth_file = Path(os.environ["HALLMARK_AUTH_FILE"])
auth_file.write_text(yaml.safe_dump({"version": 2, "profiles": {"lab": {
    "type": "http", "hosts": ["127.0.0.1"], "scheme": "bearer",
    "token": {"env": "DEMO_TOKEN"}}}}))
auth_file.chmod(0o600)  # Hallmark refuses a profile file that others can change
eht.add(PRIVATE, source="notes", remote="private", auth="lab")
[remote for remote in eht.state.config["remote"] if remote["name"] == "private"]

[{'name': 'private',
  'url': 'http://127.0.0.1:34295',
  'auth': 'lab',
  'backend': 'http'}]

Only the profile name `lab` is committed. A secret written into a profile is
refused, and so is a password in a URL.

In [21]:
literal = workspace / "literal-auth.yml"
literal.write_text(yaml.safe_dump({"version": 2, "profiles": {"lab": {
    "type": "http", "hosts": ["127.0.0.1"], "scheme": "bearer", "token": TOKEN}}}))
literal.chmod(0o600)
os.environ["HALLMARK_AUTH_FILE"] = str(literal)
expect_error(lambda: eht.add(PRIVATE, source="notes", auth="lab"))
os.environ["HALLMARK_AUTH_FILE"] = str(auth_file)
with_password = PRIVATE.replace("://", "://alice:secret@")
expect_error(lambda: eht.add(with_password, source="notes"))

RemoteConfigurationError: Auth profile 'lab' field 'token' must name where its secret comes from, for example {env: VARIABLE}; secret values cannot be stored in auth.yml
RemoteConfigurationError: URLs must not contain credentials; configure them locally, for example in ~/.netrc, ~/.ssh/config or a Git credential helper


One catalogue now spans several servers. Each row names its source, each
source names its remote, and each remote names its URL and optional profile.

In [22]:
eht.commit("Catalogue Sgr A*, calibration, simulations and notes")
for remote in eht.state.config["remote"]:
    print("remote:", remote)
for source in eht.state.config["sources"]:
    print("source:", source)

remote: {'name': 'origin', 'url': 'http://127.0.0.1:38107/', 'backend': 'http'}
remote: {'name': 'dav', 'url': 'http://127.0.0.1:37061/calib/', 'backend': 'webdav'}
remote: {'name': 's3', 'url': 's3://ehtc-sims/m87/', 'backend': 's3'}
remote: {'name': 'private', 'url': 'http://127.0.0.1:34295', 'auth': 'lab', 'backend': 'http'}
source: {'name': 'm87', 'remote': 'origin', 'fmt': 'M87_{day:03d}.txt'}
source: {'name': 'sgra', 'remote': 'origin', 'fmt': 'SgrA_{day:03d}.txt'}
source: {'name': 'copy', 'remote': 'origin', 'prefix': 'copy/', 'filter': ['M87_*']}
source: {'name': 'calib', 'remote': 'dav', 'prefix': 'calib/'}
source: {'name': 'sims', 'remote': 's3', 'prefix': 'sims/'}
source: {'name': 'notes', 'remote': 'private'}


## 6. Download into the object store

Planning reads only the local catalogue. A plan must be approved before
anything is transferred.

In [23]:
plan = eht.plan_download(all_files=True)
print(plan.summary())
expect_error(lambda: eht.download(plan))

11 file(s); 62 bytes known; 7 file(s) with unknown size; estimated duration: unknown
Sources:
  origin: http://127.0.0.1:38107/ (6 file(s))
  dav: http://127.0.0.1:37061/calib/ (2 file(s))
  private: http://127.0.0.1:34295 (1 file(s))
  s3: s3://ehtc-sims/m87/ (2 file(s))
Destination: /tmp/hallmark-demo-0ftnrxgb/eht
DownloadError: Dataset downloads require explicit approval


Each file is fetched into `.hm/objects/tmp` and checked against its catalogued
checksum. It is then stored in `.hm/objects` under its SHA-1 and hard-linked into
the folder, read-only. `M87_001.txt` and `copy/M87_001.txt` hold the same
bytes, so they share one stored object (`links=3`).

In [24]:
first_download = eht.download(plan, approved=True)
print({key: first_download[key] for key in ("succeeded", "failed", "staged")})
show(eht)

{'succeeded': 11, 'failed': 0, 'staged': 11}


staged in .hm: M data.tsv 
objects: 9 file(s)
  M87_001.txt          -r--------  links=3
  M87_002.txt          -r--------  links=3
  SgrA_001.txt         -r--------  links=2
  SgrA_002.txt         -r--------  links=2
  calib/gains_001.txt  -r--------  links=2
  calib/gains_002.txt  -r--------  links=2
  copy/M87_001.txt     -r--------  links=3
  copy/M87_002.txt     -r--------  links=3
  notes_001.txt        -r--------  links=2
  sims/sim_001.h5      -r--------  links=2
  sims/sim_002.h5      -r--------  links=2


The download staged each file's SHA-1 in the catalogue but did not commit it.
`checkout` waits until it is committed, so a branch switch never loses track of
what was downloaded.

In [25]:
eht.create_branch("draft")
expect_error(lambda: eht.checkout("draft"))
eht.commit("Record downloaded checksums")

CheckoutError: Commit or unstage hallmark state changes before checkout: data.tsv (hm commit -m MESSAGE, or hm restore --staged .)


True

## 7. Keep the catalogue current

The archive gains two files and loses one. Adding the sources again updates
their rows. A file gone from the server leaves the catalogue, but your
downloaded copy stays, now untracked. Rows that did not change keep their
SHA-1.

In [26]:
for name in ["M87_003.txt", "SgrA_003.txt"]:
    (archive / name).write_text(f"EHT data {name}\n")
(archive / "SgrA_002.txt").unlink()
publish_sums(archive)
eht.add(ARCHIVE + "/M87_{day:03d}.txt", source="m87")
removed = eht.add(ARCHIVE + "/SgrA_{day:03d}.txt", source="sgra").attrs["removed"]
print("removed from the catalogue:", removed)
print("untracked:", eht.status()["untracked"])
eht.commit("Refresh the archive sources")

removed from the catalogue: ['SgrA_002.txt']
untracked: ['SgrA_002.txt']


True

A scan that fails, or that matches nothing, changes nothing.

In [27]:
before = (eht.dothm.path / "data.tsv").read_bytes()
port = servers["dav"].server_port
stop("dav")
expect_error(lambda: eht.add(DAV + "/calib/", source="calib"))
expect_error(lambda: eht.add(ARCHIVE, filter="nothing_*", source="none"))
print("catalogue unchanged:", (eht.dothm.path / "data.tsv").read_bytes() == before)
start("dav", workspace / "servers/dav", port=port);

DownloadError: Remote scan failed (previous catalogue kept): Failed to download from http://127.0.0.1: ConnectionError
ValueError: No remote files matched filter 'nothing_*' under http://127.0.0.1:38107; previous catalogue kept
catalogue unchanged: True


## 8. Checksums protect every download

The archive overwrites `M87_003.txt` after publishing its checksum. Hallmark
rejects that file, keeps the other new one, and leaves nothing partial behind.

In [28]:
(archive / "M87_003.txt").write_text("overwritten by mistake\n")
checked = eht.download(eht.plan_download(all_files=True), approved=True)
print({key: checked[key] for key in ("succeeded", "failed", "errors")})
print("left in .hm/objects/tmp:", list((eht.dothm.path / "objects/tmp").iterdir()))

{'succeeded': 1, 'failed': 1, 'errors': ['Checksum mismatch for M87_003.txt (sha256)']}
left in .hm/objects/tmp: []


Once the archive restores the file, the next download fetches only what is
still missing.

In [29]:
(archive / "M87_003.txt").write_text("EHT data M87_003.txt\n")
plan = eht.plan_download(all_files=True)
print(plan.summary().splitlines()[0], f"({len(plan.skipped_items)} already here)")
eht.download(plan, approved=True)
eht.commit("Record the new checksums")

1 file(s); 0 bytes known; 1 file(s) with unknown size; estimated duration: unknown (11 already here)


True

## 9. Your files are never overwritten

A file of your own at a catalogued path stops the whole download.

In [30]:
(eht.worktree / "SgrA_001.txt").unlink()
(eht.worktree / "SgrA_001.txt").write_text("my own notes\n")
expect_error(lambda: eht.plan_download(all_files=True))

DownloadError: Existing paths conflict with the catalog or cannot be verified:
SgrA_001.txt
Nothing downloaded. Delete these paths first to replace them.


After you remove it, the file comes back from `.hm/objects` without contacting
the server.

In [31]:
(eht.worktree / "SgrA_001.txt").unlink()
requests_before = len(sent["archive"])
restored = eht.download(eht.plan_download(all_files=True), approved=True)
print("from .hm/objects:", restored.get("from_store"),
      "| files sent by the archive:", len(sent["archive"]) - requests_before)

from .hm/objects: ['SgrA_001.txt'] | files sent by the archive: 0


The read-only links keep the store safe from ordinary edits. A downloaded file
and its stored object are the same data on disk, though, so making the file
writable and editing it in place changes the object too. Hallmark records each
object's size and modification time, so it notices: `status` lists the file,
the next download refuses to touch it, and the changed object leaves the store.

In [32]:
edited = eht.worktree / "SgrA_001.txt"
edited.chmod(0o644)
edited.write_text("edited in place\n")
modified = eht.status()["worktree"]["modified"]
print("modified:", modified)
expect_error(lambda: eht.plan_download(all_files=True))

modified: ['SgrA_001.txt']
DownloadError: Existing paths conflict with the catalog or cannot be verified:
SgrA_001.txt (edited in place; delete it to download a fresh copy)
Nothing downloaded. Delete these paths first to replace them.


Deleting the edited file lets the next download fetch a fresh copy from the
archive.

In [33]:
edited.unlink()
refetched = eht.download(eht.plan_download(all_files=True), approved=True)
print({key: refetched.get(key) for key in ("succeeded", "from_store")},
      "|", edited.read_text().strip())

{'succeeded': 1, 'from_store': None} | EHT data SgrA_001.txt


## 10. Clone the catalogue

`clone` copies the catalogue and its git history. Downloading is optional;
this clone has no files yet.

In [34]:
copy = Repo.clone("eht", "eht-copy", download=False)
print(copy.dothm.git.log("--oneline"))
show(copy)

b28a8ca Record the new checksums
f955ff0 Refresh the archive sources
12e6a14 Record downloaded checksums
2e220e4 Catalogue Sgr A*, calibration, simulations and notes
9a0bea2 Catalogue M87 from the archive
00c3537 Initial commit: local `.hm` repository
staged in .hm: nothing 
objects: 0 file(s)


These are two different kinds of remote:

- The **git remote** of `.hm` records where the history came from. It is local
  and never committed.
- The **data remotes** in `config.yml` say where the files are. They travel
  with the catalogue.

In [35]:
print("git remote: ", copy.dothm.git.config("--get", "remote.origin.url"))
print("data remotes:", [remote["name"] for remote in copy.state.config["remote"]])

git remote:  /tmp/hallmark-demo-0ftnrxgb/eht/.hm
data remotes: ['origin', 'dav', 's3', 'private']


## 11. Download from a mirror

A mirror holds the same files at the same paths. `remote_name` fetches every
selected file from it, and the committed SHA-1 values still check each file,
so a mirror cannot slip in different data. The clone's store starts empty, so
these files are transferred again.

In [36]:
shutil.copytree(archive, mirror, dirs_exist_ok=True)
copy.set_config(remote_name="mirror", remote_url=MIRROR + "/")
from_mirror = copy.download(
    copy.plan_download(sources=["m87"], remote_name="mirror"), approved=True)
print(from_mirror["succeeded"], "file(s); files sent by the mirror:",
      len(sent["mirror"]))

3 file(s); files sent by the mirror: 3


The rest comes from each file's own server. `copy/` holds the same bytes as
the M87 files just fetched, so it is placed from the store.

In [37]:
clone_result = copy.download(copy.plan_download(all_files=True), approved=True)
print({key: clone_result.get(key) for key in ("succeeded", "failed", "from_store")})
show(copy)

{'succeeded': 9, 'failed': 0, 'from_store': ['copy/M87_001.txt', 'copy/M87_002.txt']}
staged in .hm: M config.yml 
objects: 10 file(s)
  M87_001.txt          -r--------  links=3
  M87_002.txt          -r--------  links=3
  M87_003.txt          -r--------  links=2
  SgrA_001.txt         -r--------  links=2
  SgrA_003.txt         -r--------  links=2
  calib/gains_001.txt  -r--------  links=2
  calib/gains_002.txt  -r--------  links=2
  copy/M87_001.txt     -r--------  links=3
  copy/M87_002.txt     -r--------  links=3
  notes_001.txt        -r--------  links=2
  sims/sim_001.h5      -r--------  links=2
  sims/sim_002.h5      -r--------  links=2


## 12. Branches of a remote catalogue

Checking out a branch of a remote catalogue switches only the catalogue in
`.hm`. Downloaded files stay where they are.

In [38]:
eht.create_branch("day-4")
eht.checkout("day-4")
(archive / "M87_004.txt").write_text("EHT data M87_004.txt\n")
publish_sums(archive)
eht.add(ARCHIVE + "/M87_{day:03d}.txt", source="m87")
eht.download(eht.plan_download(file_paths=["M87_004.txt"]), approved=True)
eht.commit("Add M87 day 4")
eht.checkout("main")
print("M87_004.txt catalogued on main:", "M87_004.txt" in set(eht.state.data.path))
print("M87_004.txt still on disk:     ", (eht.worktree / "M87_004.txt").exists())

M87_004.txt catalogued on main: False
M87_004.txt still on disk:      True


## Other servers

These need your own accounts, so they are not run here.

```python
# SSH or SFTP; the server computes checksums that are not published
repo.add("ssh://lab-data/srv/export/", server_checksum="sha256")
# WebDAV, such as signed-in CyVerse
repo.add("webdavs://data.cyverse.org/dav/iplant/home/alice/eht/", auth="cyverse")
# S3-compatible storage (pip install hallmark[s3])
repo.add("s3://ehtc-sims/m87/", auth="osn",
         backend_options={"endpoint_url": "https://sdsc.osn.xsede.org"})
```

The matching profiles in `~/.config/hallmark/auth.yml`:

```yaml
version: 2
profiles:
  cyverse:
    type: http
    hosts: [data.cyverse.org]
    scheme: basic
    username: alice
    password: {env: CYVERSE_PASSWORD}
  osn:
    type: s3
    buckets: [ehtc-sims]
    endpoints: [https://sdsc.osn.xsede.org]
    aws_profile: osn
```

SSH logins use your OpenSSH configuration. To try the team's demo server,
connect to it as described in [demo_cli.ipynb](demo_cli.ipynb) and use
`ssh://hallmark-demo/srv/hallmark_demo_data`.

## Clean up

Stop the servers and remove the demo folder.

In [39]:
if s3_mock:
    s3_mock.stop()
for name in list(servers):
    stop(name)
os.chdir(tempfile.gettempdir())
shutil.rmtree(workspace)